In [1]:
# ==========================================
# 步驟一：安裝必要套件與掛載 Google Drive
# (點擊左側播放鍵執行，執行時會跳出授權視窗，請點選允許)
# ==========================================
print("⏳ 正在安裝 Gradio 與 AI 相關套件，請稍候約 30 秒...")
!pip install -q gradio pandas google-genai

print("☁️ 準備掛載 Google Drive 雲端硬碟...")
from google.colab import drive
drive.mount('/content/drive')

print("✅ 環境建置完成！請往下執行「步驟二」的主程式。")

⏳ 正在安裝 Gradio 與 AI 相關套件，請稍候約 30 秒...
☁️ 準備掛載 Google Drive 雲端硬碟...
Mounted at /content/drive
✅ 環境建置完成！請往下執行「步驟二」的主程式。


In [3]:
import gradio as gr
import zipfile
import json
import time
import pandas as pd
import os
import glob
import hashlib
import datetime
from google import genai
from google.genai import types

os.environ["PYTHONIOENCODING"] = "utf-8"

# ==========================================
# 1. 資料夾掃描連動 (Colab 專用精簡版)
# ==========================================
def list_subfolders(parent_path):
    if not parent_path or not os.path.exists(parent_path):
        return []
    try:
        subfolders = [os.path.join(parent_path, d) for d in os.listdir(parent_path)
                      if os.path.isdir(os.path.join(parent_path, d)) and not d.startswith('.')]
        return sorted(subfolders)
    except Exception as e:
        print(f"[系統] 無法讀取資料夾 {parent_path}: {e}")
        return []

def update_sub_dropdown(parent_folder):
    if not parent_folder:
        return gr.update(choices=[], value=None, label="📂 2. 請先選擇上一層資料夾")
    new_choices = list_subfolders(parent_folder)
    if not new_choices:
        return gr.update(choices=[], value=None, label="📂 2. (此目錄下無子資料夾)")
    return gr.update(choices=new_choices, value=new_choices[0], label="📂 2. 選擇子資料夾 (學生作業區)")

# ==========================================
# 2. 核心讀取與線性虛擬碼轉換 (保留真實變數，防禦過大檔案)
# ==========================================
def extract_project_json(file_path, max_size_mb=10):
    MAX_FILE_SIZE = max_size_mb * 1024 * 1024
    try:
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            if "project.json" in zip_ref.namelist():
                file_info = zip_ref.getinfo("project.json")
                if file_info.file_size > MAX_FILE_SIZE:
                    print(f"[警告] {file_path} 內的 project.json 過大，拒絕處理。")
                    return None
                with zip_ref.open("project.json") as f:
                    return json.load(f)
    except Exception as e:
        print(f"[錯誤] 讀取 {file_path} 失敗: {e}")
        return None
    return None

def get_input_readable(input_data, blocks):
    if not input_data or not isinstance(input_data, list) or len(input_data) < 2:
        return ""
    payload = input_data[1]
    if isinstance(payload, str) and payload in blocks:
        target_b = blocks[payload]
        op = target_b.get("opcode", "unknown")
        inner_fields = []
        for fk, fv in target_b.get("fields", {}).items():
            if isinstance(fv, list) and len(fv) > 0:
                inner_fields.append(str(fv[0]).replace('\n', ' ').replace('\r', ' '))
        inner_inputs = []
        for ik, iv in target_b.get("inputs", {}).items():
            val = get_input_readable(iv, blocks)
            if val: inner_inputs.append(val)
        return f"[{op}: {' '.join(inner_fields + inner_inputs)}]"
    elif isinstance(payload, list):
        if len(payload) > 0:
            val_type = payload[0]
            if val_type == 12 and len(payload) > 1:
                return f"(變數:{str(payload[1]).replace(chr(10), ' ')})"
            elif val_type == 13 and len(payload) > 1:
                return f"(清單:{str(payload[1]).replace(chr(10), ' ')})"
            elif len(payload) > 1:
                return f"'{str(payload[1]).replace(chr(10), ' ')[:50]}'"
            else:
                return f"'{str(payload[0])[:50]}'"
    return str(payload).replace('\n', ' ').replace('\r', ' ')[:50]

def parse_chain_recursive(block_id, indent_level, blocks_dict, visited=None):
    if visited is None: visited = set()
    chain_text = ""
    current_id = block_id
    indent = "  " * indent_level

    while current_id and current_id in blocks_dict:
        if current_id in visited:
            chain_text += f"{indent}--> (警告：偵測到無限迴圈，中斷解析)\n"
            break
        visited.add(current_id)

        b = blocks_dict[current_id]
        opcode = b.get("opcode")
        if not opcode:
            current_id = b.get("next")
            continue

        param_pairs = []
        substacks = {}

        if "fields" in b:
            for key, val in b["fields"].items():
                if isinstance(val, list) and len(val) > 0:
                    param_pairs.append(f"{key}:'{str(val[0]).replace(chr(10), ' ')[:50]}'")

        if "inputs" in b:
            for key, val in b["inputs"].items():
                if key in ["SUBSTACK", "SUBSTACK2"]:
                    if len(val) > 1 and isinstance(val[1], str):
                        substacks[key] = val[1]
                else:
                    readable = get_input_readable(val, blocks_dict)
                    if readable: param_pairs.append(f"{key}:{readable[:100]}")

        param_str = ", ".join(param_pairs)
        chain_text += f"{indent}{opcode} ({param_str})\n"

        if "SUBSTACK" in substacks:
            chain_text += parse_chain_recursive(substacks["SUBSTACK"], indent_level + 1, blocks_dict, visited)
            if "SUBSTACK2" in substacks and opcode == "control_if_else":
                chain_text += f"{indent}--> (ELSE):\n"
                chain_text += parse_chain_recursive(substacks["SUBSTACK2"], indent_level + 1, blocks_dict, visited)
            if opcode.startswith("control_"):
                 chain_text += f"{indent}--> (END {opcode})\n"

        current_id = b.get("next")
    return chain_text

def clean_json_for_ai(raw_json):
    if not raw_json: return ""
    pseudo_code = ""

    for target in raw_json.get("targets", []):
        target_name = str(target.get('name', '未命名')).replace('\n', ' ')
        pseudo_code += f"\n[角色/背景：{target_name}]\n"

        variables = target.get("variables", {})
        lists = target.get("lists", {})
        if variables or lists:
            pseudo_code += "  ▶ 變數與清單定義：\n"
            for v_id, v_val in variables.items():
                if isinstance(v_val, list) and len(v_val) > 1:
                    safe_name = str(v_val[0])
                    safe_value = str(v_val[1]).replace('\n', ' ').replace('\r', ' ')[:30]
                    pseudo_code += f"    - [定義] {safe_name} = {safe_value}\n"
            for l_id, l_val in lists.items():
                if isinstance(l_val, list) and len(l_val) > 1:
                    safe_name = str(l_val[0])
                    list_len = len(l_val[1]) if len(l_val)>1 and isinstance(l_val[1], list) else 0
                    pseudo_code += f"    - [定義] {safe_name} (長度: {list_len})\n"

        blocks = target.get("blocks", {})
        VALID_START_PREFIXES = ("event_", "procedures_definition", "control_start_as_clone")

        top_levels = [b_id for b_id, b_val in blocks.items() if isinstance(b_val, dict) and b_val.get("topLevel") and b_val.get("opcode", "").startswith(VALID_START_PREFIXES)]

        for start_id in top_levels:
            pseudo_code += "  ▶ 執行序列：\n"
            pseudo_code += parse_chain_recursive(start_id, 2, blocks)

    return pseudo_code

def extract_json_from_text(raw_text):
    clean_text = raw_text.replace("```json", "").replace("```", "").strip()
    start_idx = clean_text.find('{')
    end_idx = clean_text.rfind('}')
    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        return clean_text[start_idx:end_idx+1]
    return clean_text

# ==========================================
# 3. Prompt
# ==========================================
def generate_grading_prompt(theme, rules, template_clean_code, example_clean_code):
    template_section = f"【初始空白範本】\n{template_clean_code}\n" if template_clean_code else "【初始空白範本】\n無\n"
    example_section = f"【老師參考解答】\n{example_clean_code}\n" if example_clean_code else "【老師參考解答】\n無\n"

    return f"""任務：你是一位懂得欣賞學生創意的 Scratch 資深教師，請批改作業【{theme}】。

【老師設定的評分法律】
{rules}

{template_section}
{example_section}

🔥【評分嚴格度指示】🔥
1. 鼓勵多元演算法：只要「最終執行邏輯」與題目要求完全一致，就算用了與老師不同的積木寫法，也給滿分。
2. 嚴禁同情分：如果邏輯或防呆機制不完整，必須嚴格依照法律扣分。
3. 加分題「絕對不扣分」原則：加分項目是額外獎勵！若達成請給予加分，若沒做或做錯，絕對不可列入扣分項目。
4. 抄襲與空白判定：
   - 若學生的程式碼與【初始空白範本】完全一致，給 0 分。
   - 若與【老師參考解答】完全一致，代表寫出了標準答案，必須給予滿分，不可扣分！"""

# ==========================================
# 4. 單一 AI 批改核心
# ==========================================
def single_agent_grading(api_keys_raw, rules, theme, clean_code, template_code, example_code, model_name):
    api_keys = []
    for k in api_keys_raw:
        if k and k.strip():
            safe_key = k.strip().encode('ascii', 'ignore').decode('ascii')
            if safe_key: api_keys.append(safe_key)

    if not api_keys:
        return {"score": 0, "comments": "未提供有效的 API Key", "deducted_items": "錯誤", "creative_highlights": "無"}

    system_prompt = generate_grading_prompt(theme, rules, template_code, example_code)
    current_key_idx = 0

    grading_schema = types.Schema(
        type=types.Type.OBJECT,
        properties={
            "logic_analysis": types.Schema(type=types.Type.STRING, description="深度邏輯分析，必須明確指出對錯"),
            "creative_highlights": types.Schema(type=types.Type.STRING, description="說明創意亮點，無則填'無'"),
            "score": types.Schema(type=types.Type.INTEGER, description="整數 (0~110，包含bonus)"),
            "comments": types.Schema(type=types.Type.STRING, description="給學生的講評"),
            "deducted_items": types.Schema(type=types.Type.STRING, description="扣分原因，無則填'無'")
        },
        required=["logic_analysis", "creative_highlights", "score", "comments", "deducted_items"]
    )

    def ask_agent(prompt, user_text, temp):
        nonlocal current_key_idx
        max_attempts = len(api_keys)

        for attempt in range(max_attempts):
            client = genai.Client(api_key=api_keys[current_key_idx])
            try:
                contents = [
                    types.Content(role="user", parts=[types.Part.from_text(text=prompt)]),
                    types.Content(role="model", parts=[types.Part.from_text(text="收到，請提供代碼。")]),
                    types.Content(role="user", parts=[types.Part.from_text(text=user_text)])
                ]

                response = client.models.generate_content(
                    model=model_name,
                    contents=contents,
                    config=types.GenerateContentConfig(
                        temperature=temp,
                        response_mime_type="application/json",
                        response_schema=grading_schema
                    )
                )

                json_str = extract_json_from_text(response.text.strip())
                return json.loads(json_str, strict=False)

            except json.JSONDecodeError as je:
                raise Exception(f"模型產生了無效的 JSON 字串: {str(je)}")
            except Exception as e:
                error_msg = str(e)
                if "429" in error_msg:
                    if attempt < max_attempts - 1:
                        print(f"[系統] API Key {current_key_idx+1} 額度耗盡！自動切換...")
                        current_key_idx = (current_key_idx + 1) % len(api_keys)
                        time.sleep(1)
                        continue
                    else:
                        raise Exception("所有 API Key 額度已耗盡！")
                raise Exception(f"API 錯誤 ({model_name}): {error_msg}")

    try:
        return ask_agent(f"[助教指令]\n{system_prompt}", f"[受測代碼]\n{clean_code}", temp=0.0)
    except Exception as e:
        return {"score": 0, "comments": f"系統異常: {str(e)}", "deducted_items": "錯誤", "creative_highlights": "無"}

# ==========================================
# 5. UI 輔助與動態延遲控制
# ==========================================
def chat_with_ai_assistant(api_key_1, api_key_2, model_name, user_msg, chat_history, rules):
    api_keys = [k for k in [api_key_1, api_key_2] if k]
    if not api_keys:
        chat_history.append((user_msg, "⚠️ 請先填寫 API Key！"))
        return chat_history, ""

    prompt = f"你是一個專為資訊老師服務的 Scratch 批改規則修訂助手。請輸出純文字條列格式。\n目前規則：\n{rules}\n老師指正：{user_msg}"

    try:
        client = genai.Client(api_key=api_keys[0].strip().encode('ascii', 'ignore').decode('ascii'))
        response = client.models.generate_content(model=model_name, contents=prompt)
        ai_reply = response.text
    except Exception as e:
        ai_reply = f"❌ 連線失敗 ({model_name})：{str(e)}"

    chat_history.append((user_msg, ai_reply.strip()))
    return chat_history, ""

def apply_chat_to_rules(chat_history, current_rules):
    if chat_history and len(chat_history) > 0: return chat_history[-1][1]
    return current_rules

def export_rules_to_file(rules_text, theme_text):
    date_str = datetime.datetime.now().strftime("%Y%m%d")
    safe_theme = str(theme_text).strip().replace("/", "_").replace("\\", "_") if theme_text else "未命名作業"
    file_name = f"{safe_theme}_規則_{date_str}版本.txt"
    file_path = os.path.join(os.getcwd(), file_name)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(rules_text)
    return file_path

def load_rules_from_txt(file_obj):
    if file_obj is None: return ""
    with open(file_obj.name, "r", encoding="utf-8") as f: return f.read()

def test_teacher_example(api_key_1, api_key_2, model_name, theme, rules, template_upload, example_upload, max_size_mb):
    api_keys = [api_key_1, api_key_2]
    if not any(k.strip() for k in api_keys if k): yield "⚠️ 請先填寫至少一組 API Key！", "" ; return
    if not example_upload: yield "⚠️ 請先上傳要測試的檔案！", "" ; return

    clean_code = clean_json_for_ai(extract_project_json(example_upload.name, max_size_mb))
    template_clean_code = clean_json_for_ai(extract_project_json(template_upload.name, max_size_mb)) if template_upload else ""

    yield f"⏳ 正在啟動單一助教 ({model_name}) 進行試評...", clean_code

    res_dict = single_agent_grading(api_keys, rules, theme, clean_code, template_clean_code, clean_code, model_name)

    report = f"🎯 最終分數：{res_dict.get('score')} 分\n"
    report += f"🧠 深度邏輯分析：{res_dict.get('logic_analysis', '無')}\n"
    report += f"💡 創意亮點：{res_dict.get('creative_highlights', '無')}\n"
    report += f"📝 鼓勵性講評：{res_dict.get('comments')}\n"
    report += f"📉 扣分項目：{res_dict.get('deducted_items')}"

    yield report, clean_code

def auto_adjust_delay(selected_model):
    delay_map = {
        "gemini-3.1-flash-lite-preview": 5,
        "gemini-2.5-flash-lite": 7,
        "gemini-3.0-flash": 13,
        "gemini-2.5-flash": 13,
    }
    return gr.update(value=delay_map.get(selected_model, 15))

# ==========================================
# 6. 全班自動批改 (單一模型 + Hash 快取)
# ==========================================
def process_homework(api_key_1, api_key_2, model_name, theme, rules, sub_folder, template_upload, example_upload, delay_seconds, max_size_mb):
    api_keys = [api_key_1, api_key_2]
    if not any(k.strip() for k in api_keys if k) or not sub_folder: yield "⚠️ 請確認 API Key 和資料夾已填寫！", None ; return

    sb3_files = glob.glob(os.path.join(sub_folder, "**", "*.sb3"), recursive=True)
    if not sb3_files: yield "⚠️ 找不到 .sb3 檔案！", None ; return

    example_clean_code = clean_json_for_ai(extract_project_json(example_upload.name, max_size_mb)) if example_upload else ""
    template_clean_code = clean_json_for_ai(extract_project_json(template_upload.name, max_size_mb)) if template_upload else ""

    template_hash = hashlib.md5(template_clean_code.encode('utf-8')).hexdigest() if template_clean_code else None
    example_hash = hashlib.md5(example_clean_code.encode('utf-8')).hexdigest() if example_clean_code else None

    results = []
    evaluation_cache = {}

    log_messages = f"📂 找到 {len(sb3_files)} 份作業。啟動巢狀邏輯解析器與 ⚡Hash 指紋抓包引擎...\n"
    yield log_messages, None

    for i, file_path in enumerate(sb3_files):
        student_id = os.path.basename(os.path.dirname(file_path)) if os.path.basename(os.path.dirname(file_path)) != os.path.basename(sub_folder) else os.path.basename(file_path).replace(".sb3", "")

        raw_json = extract_project_json(file_path, max_size_mb)
        if not raw_json:
            results.append([student_id, 0, "無", f"檔案損毀或超過老師設定的 {max_size_mb}MB 上限，無法讀取", "N/A", "❌ 壞檔/過大"])
            log_messages += f"❌ 壞檔或超過容量: {student_id}\n"
            continue

        clean_code = clean_json_for_ai(raw_json)
        code_hash = hashlib.md5(clean_code.encode('utf-8')).hexdigest()
        short_hash = code_hash[:6]

        if template_hash and code_hash == template_hash:
            yield log_messages + f"⚡ 秒殺 ({i+1}/{len(sb3_files)}): {student_id} (交白卷)...\n", None
            res_dict = {"score": 0, "comments": "程式碼與初始空白範本完全相同，你是不是忘記寫或是傳錯檔案了呢？", "deducted_items": "未作答 (交白卷)", "creative_highlights": "無"}
            status = f"📄 交白卷 (Hash:{short_hash})"

        elif example_hash and code_hash == example_hash:
            yield log_messages + f"⚡ 秒殺 ({i+1}/{len(sb3_files)}): {student_id} (完美解答)...\n", None
            res_dict = {"score": 100, "comments": "完美寫出了標準答案！邏輯非常清晰，繼續保持！", "deducted_items": "無", "creative_highlights": "與老師的解答架構一致"}
            status = f"🎯 完美解答 (Hash:{short_hash})"

        elif code_hash in evaluation_cache:
            original_student = evaluation_cache[code_hash][1]
            yield log_messages + f"⚡ 快取命中 ({i+1}/{len(sb3_files)}): {student_id} (代碼結構與 {original_student} 完全一致)...\n", None
            res_dict = evaluation_cache[code_hash][0]
            status = f"🔄 雷同拷貝 (與 {original_student} 相同 | Hash:{short_hash})"

        else:
            yield log_messages + f"⏳ 正在批改 ({i+1}/{len(sb3_files)}): {student_id} (使用 {model_name})...\n", None
            res_dict = single_agent_grading(api_keys, rules, theme, clean_code, template_code=template_clean_code, example_code=example_clean_code, model_name=model_name)
            status = f"✨ 獨立原創 (Hash:{short_hash})"

            evaluation_cache[code_hash] = (res_dict, student_id)

            if i < len(sb3_files) - 1:
                yield log_messages + f"⏱️ 冷卻防爆中 ({delay_seconds} 秒)...\n", None
                time.sleep(delay_seconds)

        current_score = res_dict.get("score", 0) if res_dict else 0
        results.append([student_id, current_score, res_dict.get("creative_highlights", "無"), res_dict.get("comments", "無"), res_dict.get("deducted_items", "無"), status])
        log_messages += f"{status}: {student_id} ({current_score}分)\n"

    df = pd.DataFrame(results, columns=["班級座號", "分數", "💡創意亮點", "講評", "扣分原因", "🔍 狀態與指紋 (Hash)"])
    csv_path = os.path.join(sub_folder, "Gemini智慧批改報告.csv")
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    yield log_messages + "\n🎉 全班批改完成！", csv_path

# ==========================================
# 7. JS 控制腳本 (暗黑模式)
# ==========================================
js_toggle_code = """
() => {
    document.documentElement.classList.toggle('dark');
    document.body.classList.toggle('dark');
    if(document.documentElement.classList.contains('dark')) {
        document.documentElement.style.colorScheme = 'dark';
    } else {
        document.documentElement.style.colorScheme = 'light';
    }
}
"""
load_js_code = """
() => {
    document.documentElement.classList.add('dark');
    document.body.classList.add('dark');
    document.documentElement.style.colorScheme = 'dark';
}
"""

# ==========================================
# 8. Gradio 介面配置 (版權宣告移至底部)
# ==========================================
with gr.Blocks(theme=gr.themes.Soft(), js=load_js_code) as app:
    # 頂部：乾淨俐落的標題與開關
    with gr.Row():
        gr.Markdown("# 🧑‍🏫 Scratch AI 智慧批改助手 (EduGrade Pro)\n輕鬆導入作業、自訂批改規則，由 AI 助教為您自動評閱全班作業。")
        dark_mode_btn = gr.Button("🌓 手動切換明暗模式", size="sm", scale=0)
        dark_mode_btn.click(None, js=js_toggle_code)

    # 內容設定區塊
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ 基礎設定")

            with gr.Row():
                api_key_1 = gr.Textbox(label="🔑 1. 主力 Gemini API Key (必填)", type="password", scale=1)
                api_key_2 = gr.Textbox(label="🔑 2. 備用 Gemini API Key (選填)", type="password", placeholder="防額度用盡，自動切換", scale=1)

            with gr.Row():
                model_dropdown = gr.Dropdown(
                    choices=[
                        ("Gemini 3.1 Flash Lite (👑 推薦主力 | 15 RPM / 500 RPD)", "gemini-3.1-flash-lite-preview"),
                        ("Gemini 2.5 Flash Lite (備用 | 10 RPM / 20 RPD)", "gemini-2.5-flash-lite"),
                        ("Gemini 3.0 Flash (🎯 狙擊槍精準 | 5 RPM / 20 RPD)", "gemini-3.0-flash"),
                        ("Gemini 2.5 Flash (精準 | 5 RPM / 20 RPD)", "gemini-2.5-flash"),
                    ],
                    value="gemini-3.1-flash-lite-preview",
                    label="🧠 3. 選擇 AI 模型"
                )

            with gr.Row():
                theme = gr.Textbox(label="4. 作業主題", value="氣泡排序法應用", scale=2)
                max_size_input = gr.Number(label="💾 5. 檔案大小上限 (MB)", value=10, precision=0, scale=1)

            rules_file_input = gr.File(label="📂 6. (選填) 上傳舊版規則 (.txt)", file_types=[".txt"])
            rules = gr.Textbox(label="📝 7. 批改規則 (老師的評分法律)", lines=8)
            template_upload = gr.File(label="📄 8. 學生初始空白範本 (.sb3)", file_types=[".sb3"])
            example_upload = gr.File(label="📄 9. 老師參考解答或受測檔案 (.sb3)", file_types=[".sb3"])

            rules_file_input.upload(fn=load_rules_from_txt, inputs=[rules_file_input], outputs=[rules])

        with gr.Column(scale=1):
            with gr.Tabs():
                with gr.TabItem("🧪 步驟一：試評與規則調校"):
                    gr.Markdown("#### 🔍 1. 測試目前規則 (建立專屬評分庫)")
                    test_btn = gr.Button("🚀 試評左側上傳的受測檔案", variant="secondary")
                    test_output = gr.Textbox(label="AI 測試報告 (包含深度分析)", lines=6, interactive=False)
                    code_preview = gr.Textbox(label="系統轉換後的【巣狀邏輯與變數文字】(給 AI 看的)", lines=4, interactive=False)

                    test_btn.click(test_teacher_example, inputs=[api_key_1, api_key_2, model_dropdown, theme, rules, template_upload, example_upload, max_size_input], outputs=[test_output, code_preview])

                    gr.Markdown("---")
                    gr.Markdown("#### 💬 2. 發現問題？與助教即時修訂規則")
                    chatbot = gr.Chatbot(label="規則調校 AI 助教聊天室", height=200)
                    with gr.Row():
                        chat_input = gr.Textbox(show_label=False, placeholder="例如：請不要死板對照積木，只要功能達到即可滿分...", scale=4)
                        chat_btn = gr.Button("傳送指正", scale=1)
                    with gr.Row():
                        apply_chat_btn = gr.Button("⬇️ 套用最新規則到左側法律區", variant="primary")
                        export_rules_btn = gr.Button("💾 匯出為規則包 (傳承與分享)", variant="secondary")
                    download_rules_file = gr.File(label="📥 這裡下載規則檔", interactive=False)

                    chat_btn.click(chat_with_ai_assistant, inputs=[api_key_1, api_key_2, model_dropdown, chat_input, chatbot, rules], outputs=[chatbot, chat_input])
                    apply_chat_btn.click(fn=apply_chat_to_rules, inputs=[chatbot, rules], outputs=[rules])
                    export_rules_btn.click(fn=export_rules_to_file, inputs=[rules, theme], outputs=[download_rules_file])

                with gr.TabItem("📦 步驟二：全班正式批改"):
                    gr.Markdown("📂 **作業來源：Google Drive (Colab)**")

                    base_path = "/content/drive/MyDrive" if os.path.exists("/content/drive/MyDrive") else "/content"
                    initial_choices = list_subfolders(base_path)
                    if base_path not in initial_choices:
                        initial_choices.insert(0, base_path)

                    root_folder = gr.Dropdown(label=f"📂 1. 選擇主目錄 ({base_path})", choices=initial_choices, value=initial_choices[0] if initial_choices else None)
                    sub_folder = gr.Dropdown(label="📂 2. 選擇子資料夾", choices=[])

                    root_folder.change(fn=update_sub_dropdown, inputs=root_folder, outputs=sub_folder)

                    delay_slider = gr.Slider(minimum=2, maximum=30, value=5, step=1, label="⏳ 全班批改間隔時間 (秒)")
                    model_dropdown.change(fn=auto_adjust_delay, inputs=model_dropdown, outputs=delay_slider)

                    start_btn = gr.Button("🔥 規則調校完成！啟動全班深度批改", variant="primary")
                    log_output = gr.Textbox(label="執行日誌", lines=12, interactive=False)
                    csv_output = gr.File(label="📥 下載 CSV 批改報告")

                    start_btn.click(process_homework, inputs=[api_key_1, api_key_2, model_dropdown, theme, rules, sub_folder, template_upload, example_upload, delay_slider, max_size_input], outputs=[log_output, csv_output])

# 👇 底部：低調精緻的版權宣告框
    gr.HTML("""
    <div style="text-align: right; margin-top: 30px; padding-top: 15px; border-top: 1px solid var(--border-color-primary);">
        <a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/" target="_blank">
            <img alt="創用 CC 授權條款" style="border-width:0; display: inline-block; margin-bottom: 6px;" src="https://i.creativecommons.org/l/by-nc-sa/4.0/88x31.png" />
        </a>
        <br />
        <span style="font-size:0.85em; color: var(--body-text-color-subdued);">
            本專案採用 <a rel="license" href="http://creativecommons.org/licenses/by-nc-sa/4.0/deed.zh_TW" target="_blank" style="color: var(--body-text-color-subdued); text-decoration:underline;">創用 CC 姓名標示-非商業性-相同方式分享 4.0 國際</a> 授權。<br>
            由教育工作者開發，歡迎教育界先進複製、修改與分享，但<b>嚴禁任何商業用途</b>。
        </span>
    </div>
    """)

app.launch(share=True, debug=True)

/tmp/ipykernel_10930/263274320.py:436: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), js=load_js_code) as app:
/tmp/ipykernel_10930/263274320.py:436: DeprecationWarning: The 'js' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'js' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), js=load_js_code) as app:
/tmp/ipykernel_10930/263274320.py:487: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="規則調校 AI 助教聊天室", height=200)
/tmp/ipykernel_10930/263274320.py:487: DeprecationWarning: The default value of 

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://14b4fd18119a6afa25.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://14b4fd18119a6afa25.gradio.live
